In [1]:
import pandas as pd
import os
import glob
from datetime import datetime

# Абсолютный путь (работает)
norm_dir = "C:/Users/пользователь/Desktop/python_labi/data/normalized/variant_04/"

list_of_files = glob.glob(norm_dir + "*.csv")
if not list_of_files:
    raise FileNotFoundError("Нет CSV в " + norm_dir)

latest_file = max(list_of_files, key=os.path.getctime)
print("Загружаем:", latest_file)

df = pd.read_csv(latest_file)
df['datetime_utc'] = pd.to_datetime(df['datetime_utc'])
print("Размер:", df.shape)
df.head()

Загружаем: C:/Users/пользователь/Desktop/python_labi/data/normalized/variant_04\2026-05-28_16-59-43.csv
Размер: (168, 7)


,datetime_utc,temp_c,rel_humidity_percent,precip_mm,wind_kmh,city,variant_id
0,2024-01-01 00:00:00,7.7,77,0.0,30.6,London,4
1,2024-01-01 01:00:00,7.6,77,0.0,29.5,London,4
2,2024-01-01 02:00:00,7.3,78,0.0,29.0,London,4
3,2024-01-01 03:00:00,7.0,80,0.0,26.8,London,4
4,2024-01-01 04:00:00,6.9,80,0.0,26.0,London,4


In [2]:
df['date'] = df['datetime_utc'].dt.date
print("Период:", df['date'].min(), "–", df['date'].max())

Период: 2024-01-01 – 2024-01-07


In [3]:
daily = df.groupby('date').agg(
    avg_temp_c=('temp_c', 'mean'),
    max_temp_c=('temp_c', 'max'),
    total_precip_mm=('precip_mm', 'sum'),
    rainy_hours=('precip_mm', lambda x: (x > 0).sum()),
    max_wind_kmh=('wind_kmh', 'max')
).reset_index()

print("Агрегировано строк:", daily.shape[0])
daily.head()

Агрегировано строк: 7


,date,avg_temp_c,max_temp_c,total_precip_mm,rainy_hours,max_wind_kmh
0,2024-01-01,7.745833,10.1,8.3,8,30.6
1,2024-01-02,11.037500,12.9,9.6,11,52.3
2,2024-01-03,8.391667,10.4,2.3,6,25.9
3,2024-01-04,6.825000,8.2,27.7,8,27.4
4,2024-01-05,5.700000,7.1,4.6,8,33.7


In [4]:
daily['city_id'] = 'GB_LON'

cities = pd.read_csv("configs/cities.csv")   # относительный путь из корня
print("Справочник:")
print(cities.head())

mart = daily.merge(cities[['city_id', 'city_name', 'country_code']], 
                   on='city_id', how='left')
print("После join строк:", len(mart))
mart.head()

Справочник:
  city_id        city_name country_code  latitude  longitude          timezone
0  RU_MOW           Москва           RU   55.7558    37.6176     Europe/Moscow
1  RU_LED  Санкт-Петербург           RU   59.9311    30.3609     Europe/Moscow
2  RU_NSK      Новосибирск           RU   55.0084    82.9357  Asia/Novosibirsk
3  GB_LON           Лондон           GB   51.5072    -0.1276     Europe/London
4  US_NYC         Нью-Йорк           US   40.7128   -74.0060  America/New_York
После join строк: 7


,date,avg_temp_c,max_temp_c,total_precip_mm,rainy_hours,max_wind_kmh,city_id,city_name,country_code
0,2024-01-01,7.745833,10.1,8.3,8,30.6,GB_LON,Лондон,GB
1,2024-01-02,11.037500,12.9,9.6,11,52.3,GB_LON,Лондон,GB
2,2024-01-03,8.391667,10.4,2.3,6,25.9,GB_LON,Лондон,GB
3,2024-01-04,6.825000,8.2,27.7,8,27.4,GB_LON,Лондон,GB
4,2024-01-05,5.700000,7.1,4.6,8,33.7,GB_LON,Лондон,GB


In [5]:
top5 = mart.nlargest(5, 'max_wind_kmh')[['date', 'max_wind_kmh', 'city_name']]
print("Топ-5 дней по скорости ветра:")
print(top5)

Топ-5 дней по скорости ветра:
         date  max_wind_kmh city_name
1  2024-01-02          52.3    Лондон
4  2024-01-05          33.7    Лондон
0  2024-01-01          30.6    Лондон
3  2024-01-04          27.4    Лондон
2  2024-01-03          25.9    Лондон


In [6]:
mart_dir = "data/mart/variant_04"   # относительный путь из корня
os.makedirs(mart_dir, exist_ok=True)

now_str = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
mart_path = os.path.join(mart_dir, f"mart_daily_{now_str}.csv")

mart.to_csv(mart_path, index=False, encoding='utf-8')
print(f"Витрина сохранена: {mart_path}")

Витрина сохранена: data/mart/variant_04\mart_daily_2026-05-28_18-24-57.csv
